# 05 - Contrastive Probe Inference

Runs the spatial-grounding probe as a factorial of control conditions over the
frozen constructed set, logging both the executable action and a continuous readout.

**Continuous readout.** OpenVLA emits one token per action dimension and decodes
it to a bin centre, so the executable action is quantised. The lateral bin is
roughly 1e-3 wide while observed lateral predictions sit between 1e-4 and 4e-3
of zero, which put the earlier median paired difference at exactly zero in every
stratum with several pairs bit-identical between the two instructions. Alongside
the argmax action, each prediction now records the expected bin centre under the
model's own distribution over action tokens, which resolves differences smaller
than one bin. 

**Control conditions.** A model mapping the token `left` to a leftward action without consulting the
image reproduces the expected sign flip exactly, and a model whose lateral output
ignores the image produces no difference for reasons unrelated to language. Each
condition varies one factor. Mirroring reverses the lateral axis with the
instruction held fixed, pairing an instruction with another scene removes its
referent while leaving the language intact, and removing the spatial term gives a
within-scene reference.

**Constructed scenes.** The experiments run on the constructed set, which holds
two instances of the target noun and whose geometry is recorded rather than
inferred. The probe reads the identified channel and the reliability bounds
from Notebook 03 set how much per-scene noise these paired comparisons must
absorb. The two roles remain disjoint, so no frame is both a construction base
and an identification trial.

The constructed source is read from the frozen `evaluation_set.csv` written by
Notebook 04. The manifest holds everything ever
built, including scenes that failed the approval screen and the overbuild beyond the
target, and probing those would spend GPU time on stimuli the analysis excludes
while making the set that was measured depend on when the probe happened to run.

## 1. Dependencies

OpenVLA-7B loads only against the pinned dependency set, which is installed here
rather than inherited from a session that happened to run first.

In [1]:
import subprocess
import sys
from importlib.metadata import version

REQUIRED_PYTHON = (3, 12)          # highest version with wheels for the pinned set
COLAB_RUNTIME_VERSION = '2026.07'  # last runtime version shipping that interpreter

PINS = [
    'transformers==4.40.1',
    'tokenizers==0.19.1',
    'timm==0.9.10',
    'huggingface_hub==0.23.4',
    'accelerate==0.30.1',
    'bitsandbytes>=0.45.0',
    'protobuf>=6.31.1,<7',
]
EXPECTED = {
    'transformers': '4.40.1',
    'tokenizers': '0.19.1',
    'timm': '0.9.10',
    'huggingface_hub': '0.23.4',
    'accelerate': '0.30.1',
}

assert sys.version_info[:2] == REQUIRED_PYTHON, (
    f'Python {sys.version_info.major}.{sys.version_info.minor} is active, but the pinned '
    f'dependencies require Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}. Set Runtime > '
    f'Change runtime type > Runtime version to {COLAB_RUNTIME_VERSION}, then reconnect and '
    f'run this notebook from the top.'
)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:', *PINS],
    check=True,
)

installed = {name: version(name) for name in EXPECTED}
for name, found in installed.items():
    print(f'{name:16s} {found}')

mismatched = {n: v for n, v in installed.items() if v != EXPECTED[n]}
assert not mismatched, (
    'Installed versions differ from the pin: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in mismatched.items())
    + '. Re-run this cell and check the install reported no error.'
)

imported = {n: getattr(sys.modules[n], '__version__', '') for n in EXPECTED
            if n in sys.modules}
stale = {n: v for n, v in imported.items() if v and v != EXPECTED[n]}
assert not stale, (
    'These packages were imported before the install and the session is still '
    'running the earlier code: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in stale.items())
    + '. Restart the session (Runtime > Restart session) and run from the top.'
)
print(f'\nPython {sys.version.split()[0]}; pinned dependency set active')

transformers     4.40.1
tokenizers       0.19.1
timm             0.9.10
huggingface_hub  0.23.4
accelerate       0.30.1

Python 3.12.13; pinned dependency set active


## 2. Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/v2/bridge'
CONSTRUCTED_DIR = '/content/drive/MyDrive/openvla_cache/v2/constructed'
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv'
print('cache       ->', CACHE_DIR)
print('constructed ->', CONSTRUCTED_DIR)
print('log         ->', PROBE_CSV)

Mounted at /content/drive
cache       -> /content/drive/MyDrive/openvla_cache/v2/bridge
constructed -> /content/drive/MyDrive/openvla_cache/v2/constructed
log         -> /content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv


## 3. Import the code

Clones the project code from GitHub into the runtime and imports the loader,
inference, control, and logging functions from there, so the code always matches
the pushed commit.

In [3]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('action_bins', 'prediction_log', 'model', 'data', 'controls', 'compose_scenes', 'export_pairs', 'detect_duplicates', 'analysis',):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

from model import (load_openvla, predict_action, predict_action_dist,
                   describe_action_space, verify_readout, run_metadata,
                   append_prediction_log)
from prediction_log import ensure_readable
from data import AXIS_INDEX, AXIS_LATERAL
from controls import (plan_stimuli, build_scene_swap, apply_image_transform,
                      strip_spatial_term, DEFAULT_CONDITIONS)
from compose_scenes import (evaluation_scenes, load_constructed_manifest,
                            resolve_scene_arrangement)
import analysis
print(f'imported project modules from {module_dir} @ {commit}')

imported project modules from /content/ECS8056 @ 050bcc0


## 4. Load OpenVLA-7B

In [4]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)
print(meta)

[load_openvla] GPU: NVIDIA A100-SXM4-40GB (sm_80, 39.5 GB) | precision=bf16 | attn=eager | 4bit=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[load_openvla] Loaded. GPU memory allocated: 4.08 GB
{'gpu_name': 'NVIDIA A100-SXM4-40GB', 'gpu_capability': 'sm_80', 'dtype': 'bfloat16', 'seed': 42, 'torch': '2.11.0+cu128', 'transformers': '4.40.1', 'bitsandbytes': '0.50.2'}


## 5. Action space and readout constants

The continuous readout reimplements the decoding OpenVLA performs inside
`predict_action`, against constants that live in its remote modelling code and
can move between revisions. Those constants are printed here rather than
assumed. The reimplementation is then required to reproduce the executable action
exactly on real inputs before any continuous value is used, which is the gate in
section 8. It runs there rather than here because it draws its inputs from the
probe set, so that what is verified is the stimuli the sweep will actually use.

`bin_width` is the resolution floor of the argmax readout on each dimension. Two
predictions closer together than this cannot differ in the executable action.

In [5]:
space = describe_action_space(vla)
for key, value in space.items():
    if isinstance(value, list):
        print(f'{key:20} ' + ' '.join(f'{v:+.5f}' if isinstance(v, float) else str(v)
                                      for v in value))
    else:
        print(f'{key:20} {value}')
print()
lateral_index = AXIS_INDEX[AXIS_LATERAL]
print(f"lateral (component {lateral_index}) bin width: "
      f"{space['bin_width'][lateral_index]:.6f}")

unnorm_key           bridge_orig
action_dim           7
n_bins               255
vocab_size           32000
bin_center_first     -0.996078431372549
bin_center_last      0.996078431372549
action_token_id_min  31745
action_token_id_max  31999
q01                  -0.02873 -0.04170 -0.02609 -0.08092 -0.09289 -0.20718 +0.00000
q99                  +0.02831 +0.04086 +0.04016 +0.08192 +0.07793 +0.20383 +1.00000
mask                 True True True True True True False
bin_width            +0.00022 +0.00033 +0.00026 +0.00064 +0.00067 +0.00162 +0.00394

lateral (component 1) bin width: 0.000325


## 6. Assemble the probe set

The constructed scenes come from the frozen `evaluation_set.csv`. A missing frozen set stops the run here.

The axis is lateral by construction, and three fields carry what construction
recorded. `base_scene_id`, the frame the stimulus was composited from, and the
target sides in image coordinates.

The blind agreement labels from Notebook 04 are consulted before a scene enters
the sweep. A scene hand-labelled `unclear` is excluded. A scene whose hand label
disagrees with the recorded arrangement is probed under the human reading,
because the arrangement is defined relative to the real arm and the annotator
saw the arm where the detector may not have. Its target signs are re-derived
from the human arrangement and the instances' relative order. The recorded manifest columns are left as recorded, so the
disagreement stays visible.

The geometry is logged unconverted. The constant relating image position to the
sign of the lateral action (`IMAGE_X_TO_LATERAL_SIGN`) was identified
empirically in Notebook 03, and it is applied in [analysis.py](analysis.py). `base_scene_id` is logged because the frozen set draws each
same-side scene and its `opposite` counterpart from one frame, without it the log
cannot express the pairing, and the decisive contrast falls back to comparing two
groups of scenes when it could hold the frame fixed.

In [6]:
from collections import Counter

probe_set = []

constructed = evaluation_scenes(CONSTRUCTED_DIR)
unclear_ids = set()
relabelled = 0
for scene in constructed:
    arrangement = resolve_scene_arrangement(scene)
    if arrangement is None:
        unclear_ids.add(scene['construct_id'])
        continue
    relabelled += int(arrangement['relabelled'])
    probe_set.append({
        'scene_source': 'constructed',
        'scene_id': scene['construct_id'],
        'pair_id': scene['construct_id'],
        'base_scene_id': str(scene['base_scene_id']),
        'spatial_term': scene['spatial_term'],
        'axis': AXIS_LATERAL,
        'axis_index': AXIS_INDEX[AXIS_LATERAL],
        'configuration': arrangement['configuration'],
        'expected_sign_image': int(scene['expected_sign_image']),
        'target_sign_a_image': arrangement['target_sign_a_image'],
        'target_sign_b_image': arrangement['target_sign_b_image'],
        'category': 'constructed',
        'feasible_both': 'yes',
        'duplicate_target': 'yes',
        'image_path': os.path.join(CONSTRUCTED_DIR, scene['image_path']),
        'instr_a': scene['instr_a'],
        'instr_b': scene['instr_b'],
    })

print(f'{len(probe_set)} constructed scenes')
print(f'excluded as unclear arrangement: {len(unclear_ids)}')
print(f'probed under the hand-labelled arrangement: {relabelled}')
print('configurations:', dict(Counter(p['configuration'] for p in probe_set)))
print('base frames:', len({p['base_scene_id'] for p in probe_set}))
if not constructed:
    built = (len(load_constructed_manifest(CONSTRUCTED_DIR))
             if os.path.isfile(os.path.join(CONSTRUCTED_DIR,
                                            'constructed_manifest.csv')) else 0)
    raise RuntimeError(
        f'\nNo frozen evaluation set in {CONSTRUCTED_DIR} ({built} scenes built).)')


340 constructed scenes
excluded as unclear arrangement: 60
probed under the hand-labelled arrangement: 53
configurations: {'same_side_left': 93, 'opposite': 151, 'same_side_right': 96}
base frames: 191


## 7. Expand the condition factorial

Every scene is expanded into the predictions its applicable conditions require.
A condition is skipped, not approximated, when its precondition fails. The mirror
conditions need a lateral term because a horizontal flip leaves depth and
vertical relations unchanged, and the term-stripped conditions need a removal
that leaves a well-formed instruction.

Refusals are counted and reported here. A truncated prompt would change the
prediction for reasons unrelated to the spatial term, so producing one would
quietly corrupt the within-scene reference. Skipping instead means the neutral
conditions cover a subset of scenes.

The swapped-scene assignment is a derangement, so no scene is ever paired with
its own image and the control cannot silently degrade into the baseline. The replacement image is drawn from the constructed set, so it stays in the same visual distribution as the original.

The frozen set holds two arrangements of the same base frame by design, and a
sibling's image carries the same background, the same object, and a genuine
referent for the instruction. Passing
`base_scene_id` as the grouping excludes the whole frame rather than the single
scene.


In [7]:
swap_map = {}
for source in ('constructed',):
    scenes = [p for p in probe_set if p['scene_source'] == source]
    if len(scenes) < 2:
        continue
    ids = [p['scene_id'] for p in scenes]
    groups = {p['scene_id']: p['base_scene_id'] for p in scenes
              if p['base_scene_id']}
    swap_map.update(build_scene_swap(ids, seed=0, groups=groups))

image_lookup = {p['scene_id']: p['image_path'] for p in probe_set}

work = []
refused = Counter()
for p in probe_set:
    stimuli = plan_stimuli(p, swap_map=swap_map, conditions=DEFAULT_CONDITIONS)
    if strip_spatial_term(p['instr_a'], p['spatial_term']) is None:
        refused[p['scene_source']] += 1
    for stim in stimuli:
        work.append((p, stim))

print(f'{len(work)} predictions across {len(probe_set)} scenes')
print('per condition:', dict(Counter(s.condition for _, s in work)))
print('per scene source:', dict(Counter(p['scene_source'] for p, _ in work)))

# Reported per source rather than pooled
print('\nterm removal refused (no neutral reference for those scenes):')
for source in ('constructed',):
    total = sum(1 for p in probe_set if p['scene_source'] == source)
    if total:
        print(f"  {source:12} {refused[source]:4}/{total} "
              f"({refused[source] / total:.1%})")

print()
for source in ('constructed',):
    for p, stim in [w for w in work if w[0]['scene_source'] == source][:4]:
        print(f"  [{p['scene_id']}] {stim.condition:16} {stim.role} "
              f"{stim.image_transform:14} :: {stim.instruction}")


2720 predictions across 340 scenes
per condition: {'baseline': 680, 'neutral': 340, 'mirror': 680, 'mirror_neutral': 340, 'swapped_scene': 680}
per scene source: {'constructed': 2720}

term removal refused (no neutral reference for those scenes):
  constructed     0/340 (0.0%)

  [c000086_same_side_left_left] baseline         a original       :: pick up the potato on the left
  [c000086_same_side_left_left] baseline         b original       :: pick up the potato on the right
  [c000086_same_side_left_left] neutral          n original       :: pick up the potato
  [c000086_same_side_left_left] mirror           a mirror         :: pick up the potato on the left


## 8. Readout correctness gate

Real stimuli are run through both paths and the results compared. 

The sample is drawn from the probe set itself. The continuous readout is a
reimplementation of OpenVLA's decoding.


In [8]:
from PIL import Image

GATE_PER_SOURCE = {'constructed': 50}
GATE_MINIMUM = 20  

def spaced(scenes, wanted):
    """`wanted` scenes spaced evenly through `scenes`.

    Spaced rather than taken from the start, so the sample is not confined to one
    scene category or one base frame.
    """
    step = max(len(scenes) // wanted, 1) if wanted else 1
    return scenes[::step][:wanted]


picked = [p for source, wanted in GATE_PER_SOURCE.items()
          for p in spaced([q for q in probe_set
                           if q['scene_source'] == source], wanted)]

taken = {(p['scene_source'], p['scene_id']) for p in picked}
for p in probe_set:
    if len(picked) >= GATE_MINIMUM:
        break
    if (p['scene_source'], p['scene_id']) not in taken:
        picked.append(p)

assert len(picked) >= GATE_MINIMUM, (
    f'only {len(picked)} stimuli available for the readout gate, which is too few '
    f'to establish agreement (at least {GATE_MINIMUM} are needed). The probe set '
    f"holds {dict(Counter(p['scene_source'] for p in probe_set))}.")

gate_counts = Counter(p['scene_source'] for p in picked)
gate_samples = [(Image.open(p['image_path']), p['instr_a']) for p in picked]
gate = verify_readout(processor, vla, gate_samples)
assert gate['matched'] == gate['checked'], gate
print('continuous readout verified against predict_action on '
      f"{gate['checked']} stimuli: {dict(gate_counts)}")


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[verify_readout] 50/50 exact | max deviation 0.000e+00 | min action-token mass 0.0000
continuous readout verified against predict_action on 50 stimuli: {'constructed': 50}


## 9. Start the log

The log is written fresh for this generation. Nothing is migrated from an earlier
one.

The file is created with the complete header from `probe_log_fields()` in
[prediction_log.py](prediction_log.py), the single declaration of the log's
columns, which the probe loop and the analysis notebooks also read.

In [9]:
import pandas as pd
from prediction_log import PROBE_EXTRA_FIELDS, probe_log_fields

PROBE_LOG_FIELDS = probe_log_fields()
print(f'v4 schema: {len(PROBE_LOG_FIELDS)} columns')

os.makedirs(os.path.dirname(PROBE_CSV), exist_ok=True)
existing = pd.read_csv(PROBE_CSV) if os.path.exists(PROBE_CSV) else None
if existing is not None and len(existing) == 0:
    existing = None
    print('replacing the empty log with the current header')

if existing is not None:
    print(f'{PROBE_CSV} holds {len(existing)} rows, the probe below resumes into it')
    print('by scene source:', dict(existing['scene_source'].value_counts()))
else:
    pd.DataFrame(columns=PROBE_LOG_FIELDS).to_csv(PROBE_CSV, index=False)
    print(f'created {PROBE_CSV} with the full header')

v4 schema: 67 columns
/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv holds 2720 rows, the probe below resumes into it
by scene source: {'constructed': np.int64(2720)}


## 9b. Check the log is readable

`ensure_readable` reports the field counts actually present and
rebuilds the file when they differ, reading each row under the schema matching its
width so that no value moves to a different column.

It refuses to rewrite a log holding rows it cannot account for, since those would
be dropped, so the repair cannot lose data. Nothing is re-run on GPU. 

In [10]:
ensure_readable(PROBE_CSV)

/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv
  67 columns, 2720 rows, consistent


{'status': 'consistent',
 'path': '/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv',
 'report': {'header_fields': 67,
  'header': ['timestamp',
   'instruction',
   'unnorm_key',
   'do_sample',
   'gpu_name',
   'gpu_capability',
   'dtype',
   'seed',
   'torch',
   'transformers',
   'bitsandbytes',
   'a0',
   'a1',
   'a2',
   'a3',
   'a4',
   'a5',
   'a6',
   'c0',
   'c1',
   'c2',
   'c3',
   'c4',
   'c5',
   'c6',
   'b0',
   'b1',
   'b2',
   'b3',
   'b4',
   'b5',
   'b6',
   'p0',
   'p1',
   'p2',
   'p3',
   'p4',
   'p5',
   'p6',
   'h0',
   'h1',
   'h2',
   'h3',
   'h4',
   'h5',
   'h6',
   'action_mass_min',
   'scene_id',
   'pair_id',
   'base_scene_id',
   'role',
   'frame',
   'scene_source',
   'condition',
   'image_transform',
   'image_scene_id',
   'configuration',
   'expected_sign_image',
   'target_sign_a_image',
   'target_sign_b_image',
   'spatial_term',
   'axis',
   'axis_index',
   'category',
   'feasible_both',
   'duplicate_t

## 10. Instrument-check pilot

The instrument check is run on a small slice so the comparison, the runner, and
the log path execute before the full sweep. 

With the instruction held fixed, mirroring the image reverses the lateral axis
of the scene. A model that reads lateral position changes the sign of its
lateral output. The slice is drawn from the constructed same-side arrangements,
where that reversal is detectable. Reflecting an `opposite` arrangement maps
the layout onto itself, so a response near the midpoint gives an antisymmetry
and an invariance both near zero.

The pilot predictions are written to the same log under the same resume key, so
the sweep in the next section skips them.

The runner is defined here and reused by the sweep, so the two cannot drift
apart in what they log. Its resume key is
`(scene_source, pair_id, frame, condition, role, image_scene_id)`. The image
identifier is part of the key because the swapped-scene condition's replacement
image is a function of the pool and the seed.

In [11]:
import csv
from PIL import Image

FRAME = 'initial'
PILOT_SCENES = 40
PILOT_CONDITIONS = ('baseline', 'mirror', 'neutral', 'mirror_neutral')
PILOT_MIN_FLIP_RATE = 0.5  # reported for reference, the live-channel test is section 13
PILOT_OVERRIDE = True    

def resume_key(scene_source, pair_id, condition, role, image_scene_id,
               frame=FRAME):
    """The identity of one prediction, for skipping work already logged."""
    return (scene_source, pair_id, frame, condition, role, image_scene_id)


done = set()
stale_axis = 0
stale_arrangement = 0
effective_configuration = {p['scene_id']: p['configuration'] for p in probe_set}
if os.path.exists(PROBE_CSV):
    with open(PROBE_CSV, newline='') as f:
        for r in csv.DictReader(f):
            if (r.get('axis') == AXIS_LATERAL and r.get('axis_index')
                    and int(r['axis_index']) != AXIS_INDEX[AXIS_LATERAL]):
                stale_axis += 1
            if r.get('scene_source') == 'constructed':
                current = effective_configuration.get(r['scene_id'])
                if r['scene_id'] in unclear_ids or (
                        current is not None and r.get('configuration')
                        and r['configuration'] != current):
                    stale_arrangement += 1
            done.add(resume_key(r.get('scene_source', 'bridge'), r['pair_id'],
                                r.get('condition', 'baseline'), r['role'],
                                r.get('image_scene_id') or r['scene_id'],
                                frame=r.get('frame') or FRAME))
    assert not stale_axis, (
        f'{stale_axis} lateral rows in {PROBE_CSV} carry an axis_index that '
        f'predates the identified channel (component '
        f'{AXIS_INDEX[AXIS_LATERAL]}). Archive or delete the log and re-run '
        'the sweep.')
    assert not stale_arrangement, (
        f'{stale_arrangement} constructed rows in {PROBE_CSV} were logged '
        'under an arrangement that the hand labels have since changed or '
        'marked unclear. Archive or delete the log and re-run the sweep.')
    print(f'resuming: {len(done)} predictions already logged')

image_cache = {}


def load_image(scene_id):
    if scene_id not in image_cache:
        image_cache[scene_id] = Image.open(image_lookup[scene_id]).convert('RGB')
    return image_cache[scene_id]


def run_items(items, label='', report_every=100):
    """Predict and log every item not already in the log, and return the count."""
    ran = 0
    for i, (p, stim) in enumerate(items):
        key = resume_key(p['scene_source'], p['pair_id'], stim.condition,
                         stim.role, stim.image_scene_id)
        if key in done:
            continue
        image = apply_image_transform(stim.image_transform,
                                      load_image(stim.image_scene_id))
        readout = predict_action_dist(processor, vla, image, stim.instruction,
                                      compute_dtype)
        extra = {
            'scene_id': p['scene_id'],
            'pair_id': p['pair_id'],
            'base_scene_id': p['base_scene_id'],
            'role': stim.role,
            'frame': FRAME,
            'scene_source': p['scene_source'],
            'condition': stim.condition,
            'image_transform': stim.image_transform,
            'image_scene_id': stim.image_scene_id,
            'configuration': p['configuration'],
            'expected_sign_image': p['expected_sign_image'],
            'target_sign_a_image': p['target_sign_a_image'],
            'target_sign_b_image': p['target_sign_b_image'],
            'spatial_term': p['spatial_term'],
            'axis': p['axis'],
            'axis_index': p['axis_index'],
            'category': p['category'],
            'feasible_both': p['feasible_both'],
            'duplicate_target': p['duplicate_target'],
            'sample_idx': 0,
        }
        assert list(extra) == list(PROBE_EXTRA_FIELDS), (
            'log fields drifted from the schema')
        append_prediction_log(PROBE_CSV, readout.action, stim.instruction, meta,
                              readout=readout, **extra)
        done.add(key)
        ran += 1
        if report_every and ran % report_every == 0:
            print(f'{label}{ran} predictions run '
                  f'({i + 1}/{len(items)} work items seen)')
    return ran

same_side = [p for p in probe_set if p['scene_source'] == 'constructed'
             and p['configuration'].startswith('same_side')]
step = max(len(same_side) // PILOT_SCENES, 1)
pilot_scenes = {p['scene_id'] for p in same_side[::step][:PILOT_SCENES]}
pilot = [(p, s) for p, s in work
         if p['scene_id'] in pilot_scenes and s.condition in PILOT_CONDITIONS]

if not pilot:
    print('no constructed same-side scenes are available, so the pilot cannot '
            'run. The instrument check then rests on section 13, after the sweep.')
else:
    print(f'pilot: {len(pilot)} predictions over {len(pilot_scenes)} same-side '
          'scenes')
    run_items(pilot, label='pilot: ', report_every=25)

    pilot_log = pd.read_csv(PROBE_CSV)
    pilot_log = pilot_log[pilot_log['scene_id'].isin(pilot_scenes)
                          & pilot_log[f'c{AXIS_INDEX[AXIS_LATERAL]}'].notna()]
    check = analysis.mirror_check(pilot_log)
    for name in ('neutral', 'term'):
        result = check.get(name, {})
        if not result.get('n'):
            print(f'\n[{name}] no paired mirror predictions')
            continue
        print(f"\n[{name}] n={result['n']}")
        print(f"  sign flips under mirroring        : {result['flip_rate']:.1%}")
        print(f"  identical to original            : {result['identical_rate']:.1%}")
        print(f"  antisymmetry (0 if exact reversal): "
              f"median={result['antisymmetry']['median']:+.5f} "
              f"p={result['antisymmetry']['p_value']:.3g}")
        print(f"  invariance   (0 if ignored)      : "
              f"median={result['invariance']['median']:+.5f} "
              f"p={result['invariance']['p_value']:.3g}")

    decisive = check.get('neutral') if check.get('neutral', {}).get('n') else \
        check.get('term', {})
    flip_rate = decisive.get('flip_rate', 0.0)
    print(f"\npilot rates (section 13 is the live-channel test): "
          f"{flip_rate:.1%} of predictions reverse sign under reflection")

resuming: 2720 predictions already logged
pilot: 240 predictions over 40 same-side scenes

[neutral] n=40
  sign flips under mirroring        : 45.0%
  identical to original            : 0.0%
  antisymmetry (0 if exact reversal): median=-0.00147 p=0.0337
  invariance   (0 if ignored)      : median=+0.00002 p=0.757

[term] n=40
  sign flips under mirroring        : 50.0%
  identical to original            : 0.0%
  antisymmetry (0 if exact reversal): median=-0.00174 p=0.000177
  invariance   (0 if ignored)      : median=-0.00038 p=0.61

pilot rates (section 13 is the live-channel test): 45.0% of predictions reverse sign under reflection


## 11. Run the probe

The full set, run through `run_items` from the previous section. Every work
item the pilot already covered is skipped by the resume key, so nothing is
predicted twice.

A session that ends early leaves a prefix of the frozen constructed set logged,
which the resume key continues from.


In [12]:
ran = run_items(work, label='sweep: ')
print(f'probe complete: {ran} new predictions -> {PROBE_CSV}')

logged = pd.read_csv(PROBE_CSV)
print('\nrows now in the log, by source and condition:')
print(logged.pivot_table(index='scene_source', columns='condition',
                         values='role', aggfunc='count', fill_value=0))
outstanding = sum(
    1 for p, stim in work
    if resume_key(p['scene_source'], p['pair_id'], stim.condition, stim.role,
                  stim.image_scene_id) not in done)
print(f'\n{outstanding} work items still outstanding')

probe complete: 0 new predictions -> /content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv

rows now in the log, by source and condition:
condition     baseline  mirror  mirror_neutral  neutral  swapped_scene
scene_source                                                          
constructed        680     680             340      340            680

0 work items still outstanding


## 12. Determinism check

Stimuli already in the log are re-predicted. Decoding is greedy with fixed seeds,
so the repeats must agree exactly. Establishing that here means any nonzero
difference in the analysis is attributable to the manipulation rather than to
run-to-run variation.

The sample is spread across conditions rather than taken from the head of the
work list. Taking the first items would draw entirely from the constructed
baseline, leaving the transforms that build a stimulus at run time (the reflection
and the substituted image) unchecked.


In [13]:
import numpy as np

REPEAT_PER_CELL = 2  # per (scene source, condition) combination present

by_cell = {}
for p, stim in work:
    by_cell.setdefault((p['scene_source'], stim.condition), []).append((p, stim))
repeat_items = [item for items in by_cell.values()
                for item in items[:REPEAT_PER_CELL]]
print(f'{len(repeat_items)} stimuli selected across '
      f'{len(by_cell)} source and condition combinations')

repeats = []
for p, stim in repeat_items:
    image = apply_image_transform(stim.image_transform,
                                  load_image(stim.image_scene_id))
    again = predict_action_dist(processor, vla, image, stim.instruction,
                                compute_dtype)
    repeats.append((p['scene_id'], stim.condition, stim.role,
                    again.action, again.expected))

log = pd.read_csv(PROBE_CSV, float_precision='round_trip')
worst_action, worst_cont, compared = 0.0, 0.0, 0
for scene_id, condition, role, action, expected in repeats:
    match = log[(log['scene_id'] == scene_id) & (log['condition'] == condition)
                & (log['role'] == role)]
    if match.empty:
        continue
    logged_a = match[[f'a{i}' for i in range(7)]].iloc[0].to_numpy(dtype=float)
    logged_c = match[[f'c{i}' for i in range(7)]].iloc[0].to_numpy(dtype=float)
    worst_action = max(worst_action, float(np.max(np.abs(logged_a - action))))
    if np.isfinite(logged_c).all():
        worst_cont = max(worst_cont, float(np.max(np.abs(logged_c - expected))))
    compared += 1

print(f'{compared} stimuli re-predicted')
print(f'largest argmax difference:     {worst_action:.3e}')
print(f'largest continuous difference: {worst_cont:.3e}')
assert worst_action == 0.0 and worst_cont == 0.0, (
    'repeated identical inputs disagreed')
print('deterministic')

10 stimuli selected across 5 source and condition combinations


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


10 stimuli re-predicted
largest argmax difference:     0.000e+00
largest continuous difference: 0.000e+00
deterministic


## 13. Instrument check on the complete log

Whether the visual channel is live on the composited stimuli is decided here, on
the full sample, not on the pilot slice.

The same comparison as section 10, now reported per arrangement rather than
pooled. Reflecting an `opposite` arrangement maps the layout onto itself, so
the check is uninformative there by construction, and pooling it with the
same-side arrangements dilutes the stratum that carries the signal.

This check is not itself evidence of spatial language grounding. It establishes
the necessary condition that makes the rest of the analysis interpretable.


In [14]:
log = pd.read_csv(PROBE_CSV)
lateral_col = f'c{AXIS_INDEX[AXIS_LATERAL]}'
lateral = log[(log['axis_index'] == AXIS_INDEX[AXIS_LATERAL])
              & log[lateral_col].notna()]


def report(result, heading):
    if not result.get('n'):
        print(f'{heading}: no paired mirror predictions')
        return
    print(f"{heading}  n={result['n']}")
    print(f"  sign flips under mirroring       : {result['flip_rate']:.1%}")
    print(f"  identical to original            : {result['identical_rate']:.1%}")
    print(f"  mean |lateral| original          : {result['mean_abs_original']:.5f}")
    print(f"  mean |change|                    : {result['mean_abs_change']:.5f}")
    print(f"  antisymmetry (0 if exact reversal): "
          f"median={result['antisymmetry']['median']:+.5f} "
          f"p={result['antisymmetry']['p_value']:.3g}")
    print(f"  invariance   (0 if ignored)      : "
          f"median={result['invariance']['median']:+.5f} "
          f"p={result['invariance']['p_value']:.3g}")


for source in ('constructed',):
    subset = lateral[lateral['scene_source'] == source]
    if subset.empty:
        print(f'\n=== {source}: no lateral predictions ===')
        continue
    print(f'\n=== {source} ===')
    per_config = analysis.mirror_check_by_configuration(subset)
    for configuration, result in sorted(per_config.items()):
        for name in ('neutral', 'term'):
            report(result.get(name, {}),
                   f"[{configuration or 'unaltered'} / {name}]")



=== constructed ===
[opposite / neutral]  n=151
  sign flips under mirroring       : 49.0%
  identical to original            : 0.0%
  mean |lateral| original          : 0.00349
  mean |change|                    : 0.00700
  antisymmetry (0 if exact reversal): median=-0.00118 p=0.000247
  invariance   (0 if ignored)      : median=+0.00046 p=0.288
[opposite / term]  n=151
  sign flips under mirroring       : 41.7%
  identical to original            : 0.0%
  mean |lateral| original          : 0.00362
  mean |change|                    : 0.00716
  antisymmetry (0 if exact reversal): median=-0.00182 p=4.08e-06
  invariance   (0 if ignored)      : median=+0.00010 p=0.737
[same_side_left / neutral]  n=93
  sign flips under mirroring       : 61.3%
  identical to original            : 0.0%
  mean |lateral| original          : 0.00357
  mean |change|                    : 0.00807
  antisymmetry (0 if exact reversal): median=-0.00108 p=0.0346
  invariance   (0 if ignored)      : median=+0.00219 